In [1]:
import sqlite3
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

In [2]:
SQLITE_PATH = "dev_ai_articles_full.sqlite"


dev_articles = "articles"
DEV_COLUMN_CONFIG = {
    "id": "id",           # primary key column
    "title": "title",     # article title column
    "body": "body_text",       # article body/content column (set to None if not available)
}

hashnode_articles = "hashnode_articles"
HASHNODE_COLUMN_CONFIG = {
    "id": "id",           # primary key column
	"title": "title",     # article title column
	"body": "body",       # article body/content column (set to None if not available)
}

# Output table where results will be written
OUTPUT_TABLE = "article_classifications"

# How much body text to use (tokens are expensive; first 500 chars is usually enough)
BODY_PREVIEW_CHARS = 1500

# Batch size for zero-shot classification (lower if you run out of RAM)
CLASSIFICATION_BATCH_SIZE = 32

In [3]:
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

In [ ]:
id_col = DEV_COLUMN_CONFIG["id"]
title_col = DEV_COLUMN_CONFIG["title"]
body_col = DEV_COLUMN_CONFIG["body"]

query = f"SELECT {id_col}, {title_col}, {body_col} FROM {dev_articles}"

df = pd.read_sql_query(query, conn)


hashnode_df = pd.read_sql_query(f"SELECT {id_col}, {title_col}, {body_col} FROM {hashnode_articles}", conn)




DatabaseError: Execution failed on sql 'SELECT id, title, body_text FROM articles': no such table: articles

In [ ]:
df

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
3019,3363277,Hardware Hacking 101 Village - Post-Mortem,title: Hardware Hacking 101 Village Post Morte...
3020,3363288,FE/BE - Unite Them!,tl;dr; Teams should agree upon and understand ...
3021,3363939,Announcing the Colab MCP Server: Connect Any A...,When you’re prototyping locally with AI agents...
3022,3364128,I Built a Claude Code Agent That Doesn't Need ...,title: I Built a Claude Code Agent That Doesn'...


In [ ]:
hashnode_df

,id,title,body_text
0,69c4b8a8efeaf33e6b3675be,MonALISA : A Distributed Monitoring Service Ar...,Adaptive Monitoring for Large Scale Grids: a S...
1,69c4bd952d879655ece508e5,The Sweet Spot of AI Orchestration: From AWS L...,The Sweet Spot of AI Orchestration: From AWS L...
2,69c4aaa5bbdb1bc33f100e25,"45 Claude Code Hooks for Code Quality, Securit...",45 Claude Code Hooks I Use to Automate Code Qu...
3,69c4a9faaed4ec64674bd0db,Claude Code Channels: Set Up a 24/7 AI Agent v...,Claude Code Channels Just Dropped — Here's How...
4,69c4a88eea32df99834d3704,I Built My Own OpenClaw Alternative With Claud...,I Built My Own OpenClaw Alternative With Claud...
...,...,...,...
22643,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
22644,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
22645,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
22646,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


In [ ]:
df_all = pd.concat([df, hashnode_df], ignore_index=True)

In [ ]:
df_all

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
25667,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
25668,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
25669,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
25670,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


## Step 1: Embed articles

In [ ]:
df_all["combined_text"] = (
	df_all[title_col].fillna("") + " " +
	df_all[body_col].fillna("").str[:BODY_PREVIEW_CHARS]
).str.strip()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, fast, good quality

df_test = df_filtered.head(1000).copy()
texts = df_test["combined_text"].tolist()
embeddings = model.encode(
	texts,
	batch_size=64,
	show_progress_bar=True,
	convert_to_numpy=True,
)
print(f"✅ Embeddings shape: {embeddings.shape}")


embeddings

c:\Users\abbhi\anaconda3\envs\CIS4930\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches: 100%|██████████| 402/402 [28:11<00:00,  4.21s/it]


✅ Embeddings shape: (25672, 384)


array([[-0.03045662, -0.02122168, -0.02453277, ..., -0.03061731,
         0.06307959,  0.06322043],
       [-0.03914866, -0.03178781,  0.001767  , ..., -0.01439707,
        -0.03717563,  0.06547669],
       [-0.01395674, -0.05800174,  0.05732978, ...,  0.06991705,
        -0.03069576,  0.03327543],
       ...,
       [ 0.0204586 ,  0.05308988,  0.05026504, ...,  0.00267067,
        -0.02624441, -0.02031039],
       [ 0.02364533, -0.07605472,  0.0279224 , ...,  0.04681316,
        -0.05939626, -0.01434456],
       [-0.04372468, -0.0524913 , -0.02987417, ..., -0.06269559,
         0.07843874, -0.06325853]], shape=(25672, 384), dtype=float32)

## Step 2: Find optimal number of clusters (elbow + silhouette)

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Test a range of k values
k_range = range(5, 26, 2)
inertias = []
silhouette_scores = []

for k in tqdm(k_range, desc="Testing k values"):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3)
    lbls = km.fit_predict(embeddings)
    inertias.append(km.inertia_)
    # Silhouette is expensive on large N — sample 5k if needed
    sample_size = min(5000, len(embeddings))
    idx = np.random.choice(len(embeddings), sample_size, replace=False)
    silhouette_scores.append(silhouette_score(embeddings[idx], lbls[idx], sample_size=sample_size))

import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(list(k_range), inertias, 'bo-')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')

ax2.plot(list(k_range), silhouette_scores, 'ro-')
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score (higher = better)')

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f"Best k by silhouette: {best_k}")

## Step 3: Cluster with UMAP + KMeans (no PCA)

In [ ]:
import umap

# UMAP for clustering (higher n_components preserves more structure than PCA)
reducer = umap.UMAP(n_components=20, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
umap_embeddings = reducer.fit_transform(embeddings)
print(f"UMAP output shape: {umap_embeddings.shape}")

# Use best_k from silhouette analysis (or override here)
n_clusters = best_k  # change to a specific int if you want to override

km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=5)
labels = km.fit_predict(umap_embeddings)

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   Cluster {u:2d}: {c:,} articles")

### 2D UMAP visualization (for plotting only — separate from clustering)

In [ ]:
# Separate 2D UMAP just for visualization
reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
umap_2d = reducer_2d.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=labels, cmap='tab20', alpha=0.5, s=5)
plt.colorbar(scatter, label='Cluster')
plt.title('2D UMAP — Article Clusters')
plt.tight_layout()
plt.show()

## Step 4: Semantic cluster selection (no more manual picking)

In [ ]:
from sklearn.preprocessing import normalize

# Define the target theme as a natural language query
TARGET_THEME = "AI automation impact on society, environment, law, and jobs"

from sentence_transformers import SentenceTransformer
_model = SentenceTransformer("all-MiniLM-L6-v2")
theme_embedding = _model.encode([TARGET_THEME], convert_to_numpy=True)

# Compute cluster centroids in original embedding space
n_c = n_clusters
centroids = np.array([
    embeddings[labels == c].mean(axis=0) for c in range(n_c)
])

# Cosine similarity between theme and each centroid
norm_centroids = normalize(centroids)
norm_theme = normalize(theme_embedding)
cosine_sims = (norm_centroids @ norm_theme.T).flatten()

# Rank clusters by similarity
ranked = sorted(enumerate(cosine_sims), key=lambda x: -x[1])
print("Cluster similarity to target theme:")
for cluster_id, sim in ranked:
    print(f"  Cluster {cluster_id:2d}: {sim:.4f}  ({counts[cluster_id]:,} articles)")

# Auto-select top clusters above a similarity threshold
SIM_THRESHOLD = 0.3
relevant_clusters = [c for c, s in ranked if s >= SIM_THRESHOLD]
print(f"\nAuto-selected clusters: {relevant_clusters}")

In [ ]:
df_all_copy = df_all.copy()
df_all_copy["cluster"] = labels

# Print sample articles from each selected cluster for sanity check
n_samples = 5
for cluster_id in relevant_clusters:
    samples = df_all_copy[df_all_copy["cluster"] == cluster_id]["combined_text"].head(n_samples).tolist()
    print(f"\n── Cluster {cluster_id} (sim={cosine_sims[cluster_id]:.3f}) ──")
    for s in samples:
        print(f"  • {s[:120]}")

In [ ]:
df_filtered = df_all_copy[df_all_copy["cluster"].isin(relevant_clusters)].copy()
print(f"Original articles : {len(df_all_copy):,}")
print(f"Filtered articles : {len(df_filtered):,}")
print(f"Removed           : {len(df_all_copy) - len(df_filtered):,}")

## Step 5: Zero-shot classification (fixed — runs on ALL filtered articles)

In [ ]:
CANDIDATE_LABELS_PRIMARY = [
    "a coding tutorial, technical guide, or software implementation walkthrough with little discussion of societal implications",
    "an opinion piece, analysis, or commentary on AI's consequences for society, law, environment, or ethics",
]

CANDIDATE_LABELS_SECONDARY = [
    "discussion of the environmental impact or energy consumption of AI",
    "discussion of legal, regulatory, or policy issues related to AI",
    "discussion of ethical concerns, bias, or fairness in AI systems",
    "discussion of AI's impact on jobs, employment, or the economy",
    "discussion of AI in geopolitics, international competition, or national security",
    "discussion of how AI is portrayed in media or public opinion",
    "discussion of AI safety, alignment, or existential risks",
]

In [ ]:
def classify_batch(pipe, texts, candidate_labels, threshold=0.5):
    results = pipe(texts, candidate_labels, multi_label=True)
    if isinstance(results, dict):
        results = [results]
    output = []
    for r in results:
        labels_scores = [
            (label, score)
            for label, score in zip(r["labels"], r["scores"])
            if score >= threshold
        ]
        output.append(labels_scores)
    return output

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,  # CPU; change to 0 for CUDA GPU
)

In [ ]:
# Primary classification: TECHNICAL vs NON_TECHNICAL
# FIX: runs on ALL filtered articles (not just 30), uses positional indexing to avoid misalignment

texts = df_filtered["combined_text"].tolist()
primary_labels = []
primary_scores = []

for i in tqdm(range(0, len(texts), CLASSIFICATION_BATCH_SIZE), desc="Primary classification"):
    batch = texts[i : i + CLASSIFICATION_BATCH_SIZE]
    results = classify_batch(pipe, batch, CANDIDATE_LABELS_PRIMARY, threshold=0.0)

    for r in results:
        if not r or r[0][1] < 0.7:
            primary_labels.append("UNSURE")
            primary_scores.append(0.0)
            continue
        top_label, top_score = r[0]
        is_technical = "technical" in top_label.lower()
        primary_labels.append("TECHNICAL" if is_technical else "NON_TECHNICAL")
        primary_scores.append(float(top_score))

# Use .iloc-based positional assignment to avoid index misalignment
df_filtered = df_filtered.reset_index(drop=True)
df_filtered["primary_label"] = primary_labels
df_filtered["primary_score"] = primary_scores

print(df_filtered["primary_label"].value_counts())

In [ ]:
# Inspect confidence score distribution to validate the 0.7 threshold
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df_filtered["primary_score"], bins=30, edgecolor='black')
plt.axvline(0.7, color='red', linestyle='--', label='threshold=0.7')
plt.xlabel("Primary classification score")
plt.ylabel("Count")
plt.title("Distribution of primary classification confidence")
plt.legend()
plt.tight_layout()
plt.show()

## Step 6: Secondary classification (topic labeling for NON_TECHNICAL articles)

In [ ]:
non_tech_mask = df_filtered["primary_label"] == "NON_TECHNICAL"
non_tech_df = df_filtered[non_tech_mask].copy().reset_index(drop=True)
non_tech_texts = non_tech_df["combined_text"].tolist()

print(f"Non-technical articles to sub-classify: {len(non_tech_texts):,}")

# FIX: build results positionally, then merge back by index — no misalignment
secondary_labels_list = []
secondary_scores_list = []

for i in tqdm(range(0, len(non_tech_texts), CLASSIFICATION_BATCH_SIZE), desc="Secondary classification"):
    batch = non_tech_texts[i : i + CLASSIFICATION_BATCH_SIZE]
    results = classify_batch(pipe, batch, CANDIDATE_LABELS_SECONDARY, threshold=0.0)

    for r in results:
        if not r or r[0][1] < 0.7:
            secondary_labels_list.append("UNSURE")
            secondary_scores_list.append(0.0)
            continue
        top_label, top_score = r[0]
        secondary_labels_list.append(top_label)
        secondary_scores_list.append(float(top_score))

non_tech_df["secondary_label"] = secondary_labels_list
non_tech_df["secondary_score"] = secondary_scores_list

# Merge back
df_filtered = df_filtered.merge(
    non_tech_df[["secondary_label", "secondary_score"]],
    left_index=True, right_index=True, how="left"
)
df_filtered["secondary_label"] = df_filtered["secondary_label"].fillna("")
df_filtered["secondary_score"] = df_filtered["secondary_score"].fillna(0.0)

print("\nSecondary label distribution:")
print(non_tech_df["secondary_label"].value_counts())

## Step 7: LDA with lemmatization + coherence-tuned n_topics

In [ ]:
import spacy
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel

# Load spaCy for lemmatization
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Custom stopwords for AI articles
AI_STOPWORDS = {
    "ai", "model", "data", "use", "like", "also", "one", "get",
    "make", "need", "know", "way", "thing", "time", "even", "well",
    "using", "used", "new", "can", "will", "just", "work", "help",
    "artificial", "intelligence", "machine", "learning"
}

def lemmatize_text(text):
    doc = nlp(text[:5000])  # cap length for speed
    tokens = [
        token.lemma_.lower() for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token.lemma_) > 2
        and token.lemma_.lower() not in AI_STOPWORDS
    ]
    return tokens

# Lemmatize non-technical articles (the ones we care about)
nt_texts = non_tech_df["combined_text"].tolist()
print("Lemmatizing texts...")
tokenized = [lemmatize_text(t) for t in tqdm(nt_texts)]

In [ ]:
# Build gensim dictionary and corpus
id2word = corpora.Dictionary(tokenized)
id2word.filter_extremes(no_below=3, no_above=0.5)
corpus = [id2word.doc2bow(t) for t in tokenized]

# Coherence sweep to find best n_topics
topic_range = range(4, 16)
coherence_scores = []

print("Computing coherence scores across topic counts...")
for n in tqdm(topic_range):
    lda_model = gensim.models.LdaMulticore(
        corpus=corpus,
        id2word=id2word,
        num_topics=n,
        random_state=42,
        passes=5,
        workers=2
    )
    cm = CoherenceModel(model=lda_model, texts=tokenized, dictionary=id2word, coherence='c_v')
    coherence_scores.append(cm.get_coherence())

plt.figure(figsize=(8, 4))
plt.plot(list(topic_range), coherence_scores, 'go-')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score (c_v)')
plt.title('LDA Coherence — pick the elbow/peak')
plt.tight_layout()
plt.show()

best_n = list(topic_range)[np.argmax(coherence_scores)]
print(f"Best n_topics by coherence: {best_n}")

In [ ]:
# Train final LDA with best n_topics
lda_final = gensim.models.LdaMulticore(
    corpus=corpus,
    id2word=id2word,
    num_topics=best_n,
    random_state=42,
    passes=10,
    workers=2
)

print("\nTopics discovered (lemmatized, AI-stopwords removed):")
for topic_idx, topic in lda_final.print_topics(num_topics=best_n, num_words=10):
    print(f"  Topic {topic_idx}: {topic}")

In [ ]:
# Assign dominant topic to each non-technical article
doc_topics_matrix = np.array([[prob for _, prob in lda_final.get_document_topics(doc, minimum_probability=0.0)]
                               for doc in corpus])
non_tech_df = non_tech_df.copy()
non_tech_df["lda_topic"] = doc_topics_matrix.argmax(axis=1)
non_tech_df["lda_topic_prob"] = doc_topics_matrix.max(axis=1)

plt.figure(figsize=(10, 4))
non_tech_df["lda_topic"].value_counts().sort_index().plot(kind="bar")
plt.title("Document Distribution Across LDA Topics (lemmatized)")
plt.xlabel("Topic")
plt.ylabel("Number of Documents")
plt.tight_layout()
plt.show()

## Step 8: Cross-source analysis

In [ ]:
# Compare secondary label distribution by source
df_all_copy["source"] = df_all_copy.index.map(
    lambda i: "dev.to" if i < len(df) else "hashnode"
)
df_filtered["source"] = df_all_copy.loc[df_filtered.index, "source"].values

labeled = df_filtered[df_filtered["secondary_label"] != ""]
cross = labeled.groupby(["source", "secondary_label"]).size().unstack(fill_value=0)
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

cross_pct.T.plot(kind="bar", figsize=(12, 5), width=0.7)
plt.title("Secondary Label Distribution by Source (%)")
plt.xlabel("Topic")
plt.ylabel("% of articles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
cross_pct

## Step 9: Sentiment polarity per secondary label

In [ ]:
# Install if needed: pip install vaderSentiment
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "vaderSentiment", "-q"], check=True)
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = sia.polarity_scores(text[:2000])["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

non_tech_df = non_tech_df.copy()
non_tech_df["sentiment"] = non_tech_df["combined_text"].apply(vader_sentiment)

print("Sentiment distribution:")
print(non_tech_df["sentiment"].value_counts())

In [ ]:
# Sentiment breakdown per secondary label
has_label = non_tech_df[non_tech_df["secondary_label"] != ""]
sent_cross = has_label.groupby(["secondary_label", "sentiment"]).size().unstack(fill_value=0)
sent_pct = sent_cross.div(sent_cross.sum(axis=1), axis=0) * 100

sent_pct.plot(kind="bar", figsize=(12, 5), color=["#e74c3c", "#95a5a6", "#2ecc71"], width=0.7)
plt.title("Sentiment Polarity per Topic (% of articles)")
plt.xlabel("Secondary Topic Label")
plt.ylabel("% of articles")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()
sent_pct

## Step 10: Save results

In [ ]:
conn = sqlite3.connect("dev_ai_articles_full.sqlite")
df_filtered.to_sql("article_classifications", conn, if_exists="replace", index=False)
non_tech_df.to_sql("non_technical_articles", conn, if_exists="replace", index=False)
conn.close()
print("✅ Saved to SQLite")